# 11 - Expanded-data arrival-delay models and prediction

This notebook continues the earlier model flow using notebook 10 Parquet.
It compares the historical baseline, Ridge, Random Forest, Gradient-Boosted
Trees, XGBoost and CatBoost under one temporal contract.

Model selection uses December 2022 only. March and June 2023 remain locked
until a winner is frozen. The regression task is measured with MAE, RMSE,
median absolute error and p90 absolute error. A parallel classification task
predicts whether arrival delay will exceed 15 minutes and reports accuracy,
balanced accuracy, precision, recall, F1, ROC-AUC, PR-AUC and confusion counts.

In [ ]:
from pathlib import Path
import gc
import json
import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import LogisticRegression, Ridge

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.t60_modeling import (
    HistoricalMedianBaseline, MixedCategoricalRidgePreprocessor,
    add_schedule_features, compact_score, delay_classification_metrics,
    haul_direction_classification_metrics, haul_direction_segment_metrics,
    segment_metrics,
)

DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "expanded_arrival_pre_t60"
REPORT_ROOT = PROJECT_ROOT / "reports" / "expanded_models"
MODEL_ROOT = PROJECT_ROOT / "models" / "expanded"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
TARGET = "Arrival_Delay_Min"
DELAY_THRESHOLD_MINUTES = 15.0
RUN_MODELS = False
RUN_CLASSIFICATION = True
RUN_CATBOOST_CLASSIFIER = True
TRAIN_SAMPLE_PERCENT = 10
VALIDATION_SAMPLE_PERCENT = 5
TREE_SAMPLE_PERCENT = 1
SEED = 42

## 1. Load frozen splits and audit leakage

Notebook 10 must first write Parquet. Sample percentages control resources,
not dates. Test labels are loaded for final scoring only after validation
freezes the winner.

In [ ]:
def deterministic_sample(frame, percent):
    if percent >= 100:
        return frame.copy()
    bucket = pd.util.hash_pandas_object(frame["ECTRL ID"], index=False) % 100
    return frame.loc[bucket < percent].copy()

if RUN_MODELS:
    development_splits = {
        name: add_schedule_features(pd.read_parquet(DATA_ROOT / name))
        for name in ("train", "validation")
    }
    forbidden = {"ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
                 "Departure_Delay_Min", "Actual Distance Flown (nm)"}
    assert all(not (forbidden & set(frame.columns)) for frame in development_splits.values())
    cutoff = (pd.to_datetime(development_splits["train"]["FILED OFF BLOCK TIME"])
              - pd.to_datetime(development_splits["train"]["prediction_cutoff_t60"])
             ).dt.total_seconds()/60
    assert np.allclose(cutoff, 60)
    train = deterministic_sample(development_splits["train"], TRAIN_SAMPLE_PERCENT)
    validation = deterministic_sample(
        development_splits["validation"], VALIDATION_SAMPLE_PERCENT
    )
    print({
        **{name: len(frame) for name, frame in development_splits.items()},
        "locked_test_partitions_read": False,
    })
else:
    print("Set RUN_MODELS=True after notebook 10 writes Parquet.")

## 2. Shared features and historical baseline

Airports, operator and grouped aircraft are hashed for Ridge and native
categories for CatBoost. Numeric medians and scaling are learned on train.
Operational T-60 columns are included automatically if notebook 10 built them.
The model also receives origin/destination continent, transatlantic direction,
short/medium versus long-haul band, scheduled arrival hour and duration-by-direction
interactions. Separate regression and OTP15 tables are exported for <=6h, >6h,
Europe-to-Americas and Americas-to-Europe flights.

The baseline fallback is route+airline, route, departure-airport+airline,
departure airport and global median, all fitted on train.

In [ ]:
LOW_CARDINALITY_COLUMNS = [
    "STATFOR Market Segment", "Class_aircraft", "Number+Engine Type_aircraft",
    "ADEP_Continent", "ADES_Continent", "Duration_Band",
    "Transatlantic_Direction",
]
HIGH_CARDINALITY_COLUMNS = [
    "ADEP", "ADES", "AC Operator", "AC Type_grouped", "AC Registration",
]
CATEGORICAL_COLUMNS = LOW_CARDINALITY_COLUMNS + HIGH_CARDINALITY_COLUMNS
STATIC_NUMERIC_COLUMNS = [
    "Requested_FL_Imputed", "scheduled_duration_min",
    "departure_hour_sin", "departure_hour_cos",
    "departure_dow_sin", "departure_dow_cos", "departure_month",
    "scheduled_arrival_hour_sin", "scheduled_arrival_hour_cos",
    "Is_Transatlantic", "duration_x_europe_to_americas",
    "duration_x_americas_to_europe",
]
if RUN_MODELS:
    OPERATIONAL_COLUMNS = [
        column for column in train.columns
        if column.startswith(("adep_dep_", "ades_arr_", "route_arr_",
                              "operator_dep_", "operator_arr_", "rotation_",
                              "adep_scheduled_", "ades_scheduled_",
                              "scheduled_airport_pressure_"))
    ]
    NUMERIC_COLUMNS = STATIC_NUMERIC_COLUMNS + OPERATIONAL_COLUMNS
    baseline = HistoricalMedianBaseline().fit(train)
    validation_predictions = {
        "historical_baseline": baseline.predict(validation)
    }
    SELECTED_HYPERPARAMETERS = {}
    print({"numeric_features": len(NUMERIC_COLUMNS),
           "operational_features": len(OPERATIONAL_COLUMNS)})

## 3. Ridge selection

Ridge remains the primary balanced model: it is memory-efficient, stable with
high-cardinality hashing and was comparatively strong for delayed flights.
Alpha is selected on validation only. Two otherwise identical variants are
compared: the current Ridge with raw aircraft registration and an ablation
without that raw field. Rotation history and accumulated-delay variables remain
in both variants, isolating the value of the registration identifier itself.

In [ ]:
if RUN_MODELS:
    ridge_variant_columns = {
        "ridge": HIGH_CARDINALITY_COLUMNS,
        "ridge_without_registration": [
            column for column in HIGH_CARDINALITY_COLUMNS
            if column != "AC Registration"
        ],
    }
    ridge_rows, ridge_models, ridge_preprocessors = [], {}, {}
    for variant, high_cardinality_columns in ridge_variant_columns.items():
        preprocessor = MixedCategoricalRidgePreprocessor(
            LOW_CARDINALITY_COLUMNS, high_cardinality_columns, NUMERIC_COLUMNS
        )
        x_ridge_train = preprocessor.fit_transform(train)
        x_ridge_validation = preprocessor.transform(validation)
        variant_results = []
        for alpha in (0.1, 1.0, 10.0, 100.0):
            model = Ridge(alpha=alpha, solver="lsqr").fit(
                x_ridge_train, train[TARGET]
            )
            prediction = model.predict(x_ridge_validation)
            metrics = segment_metrics(
                validation[TARGET], prediction, variant, "validation"
            )
            score = compact_score(metrics)
            ridge_rows.append({"variant": variant, "alpha": alpha, **score})
            variant_results.append((score["combined_MAE_score"],
                                    score["global_MAE"], alpha, model, prediction))
        best = min(variant_results, key=lambda item: (item[0], item[1]))
        ridge_models[variant] = best[3]
        ridge_preprocessors[variant] = preprocessor
        validation_predictions[variant] = best[4]
        SELECTED_HYPERPARAMETERS[variant] = {
            "alpha": float(best[2]),
            "includes_raw_registration": variant == "ridge",
        }
        del x_ridge_train, x_ridge_validation
        gc.collect()
    ridge_selection = pd.DataFrame(ridge_rows).sort_values(
        ["variant", "combined_MAE_score", "global_MAE"]
    )
    ridge_selection.to_csv(
        REPORT_ROOT / "ridge_hyperparameter_selection.csv", index=False
    )
    display(ridge_selection)

## 4. Random Forest, GBT and XGBoost

Tree models use a smaller deterministic train sample by default because they
need more RAM, but use identical validation rows. Three focused configurations
per family vary depth/leaf regularisation and learning rate rather than running
an expensive exhaustive grid. XGBoost is the agreed fourth model.
LightGBM/SynapseML remains a future option with extra runtime complexity.

In [ ]:
def dense_tree_frame(frame, medians=None):
    categorical = frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
    hashed = FeatureHasher(
        n_features=512, input_type="string", alternate_sign=False
    ).transform(
        ([f"{column}={value}" for column, value in zip(CATEGORICAL_COLUMNS, row)]
         for row in categorical.itertuples(index=False, name=None))
    ).toarray()
    numeric = frame[NUMERIC_COLUMNS]
    medians = numeric.median() if medians is None else medians
    numeric = numeric.fillna(medians).to_numpy(dtype=np.float32)
    return np.hstack([hashed.astype(np.float32), numeric]), medians

if RUN_MODELS:
    tree_train = deterministic_sample(train, TREE_SAMPLE_PERCENT)
    x_tree_train, tree_medians = dense_tree_frame(tree_train)
    x_tree_validation, _ = dense_tree_frame(validation, tree_medians)
    tree_candidate_specs = {
        "random_forest": [
            {"n_estimators": 150, "max_depth": 12, "min_samples_leaf": 5, "max_features": 0.7},
            {"n_estimators": 250, "max_depth": 18, "min_samples_leaf": 3, "max_features": 0.8},
            {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 10, "max_features": 0.7},
        ],
        "gradient_boosted_trees": [
            {"max_iter": 250, "max_leaf_nodes": 31, "learning_rate": 0.05, "l2_regularization": 5, "min_samples_leaf": 20},
            {"max_iter": 350, "max_leaf_nodes": 31, "learning_rate": 0.03, "l2_regularization": 10, "min_samples_leaf": 20},
            {"max_iter": 250, "max_leaf_nodes": 63, "learning_rate": 0.04, "l2_regularization": 10, "min_samples_leaf": 30},
        ],
    }
    try:
        from xgboost import XGBRegressor
        tree_candidate_specs["xgboost"] = [
            {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.04, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 8},
            {"n_estimators": 700, "max_depth": 5, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.8, "reg_lambda": 12},
            {"n_estimators": 450, "max_depth": 7, "learning_rate": 0.035, "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 15},
        ]
    except ImportError:
        print("XGBoost unavailable; install project requirements.")

    tree_models, tree_tuning_rows = {}, []
    for family, candidates in tree_candidate_specs.items():
        family_results = []
        for candidate_id, parameters in enumerate(candidates, start=1):
            if family == "random_forest":
                model = RandomForestRegressor(
                    **parameters, n_jobs=2, random_state=SEED
                )
            elif family == "gradient_boosted_trees":
                model = HistGradientBoostingRegressor(
                    **parameters, random_state=SEED
                )
            else:
                model = XGBRegressor(
                    **parameters, objective="reg:absoluteerror",
                    n_jobs=2, random_state=SEED,
                )
            model.fit(x_tree_train, tree_train[TARGET])
            prediction = model.predict(x_tree_validation)
            score = compact_score(segment_metrics(
                validation[TARGET], prediction, family, "validation"
            ))
            row = {"family": family, "candidate": candidate_id,
                   "parameters": json.dumps(parameters), **score}
            tree_tuning_rows.append(row)
            family_results.append((score["combined_MAE_score"],
                                   score["global_MAE"], model, prediction, parameters))
        best = min(family_results, key=lambda item: (item[0], item[1]))
        tree_models[family] = best[2]
        validation_predictions[family] = best[3]
        SELECTED_HYPERPARAMETERS[family] = best[4]
    tree_hyperparameter_selection = pd.DataFrame(tree_tuning_rows).sort_values(
        ["family", "combined_MAE_score", "global_MAE"]
    )
    tree_hyperparameter_selection.to_csv(
        REPORT_ROOT / "tree_hyperparameter_selection.csv", index=False
    )
    display(tree_hyperparameter_selection)

## 5. CatBoost

CatBoost is the nonlinear categorical challenger. Native categories capture
route/operator/aircraft interactions without huge one-hot matrices.
Two focused depth/learning-rate configurations are compared on December.
Early stopping and time-aware fitting limit overfitting.

In [ ]:
if RUN_MODELS:
    try:
        from catboost import CatBoostRegressor, Pool
        cat_train = train.sort_values("FILED OFF BLOCK TIME").copy()
        cat_validation = validation.sort_values("FILED OFF BLOCK TIME").copy()
        cat_medians = cat_train[NUMERIC_COLUMNS].median()
        for frame in (cat_train, cat_validation):
            frame[CATEGORICAL_COLUMNS] = (
                frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
            )
            frame[NUMERIC_COLUMNS] = frame[NUMERIC_COLUMNS].fillna(cat_medians)
        cat_features = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
        train_pool = Pool(cat_train[cat_features], cat_train[TARGET],
                          cat_features=CATEGORICAL_COLUMNS)
        validation_pool = Pool(cat_validation[cat_features], cat_validation[TARGET],
                               cat_features=CATEGORICAL_COLUMNS)
        catboost_candidates = [
            {"iterations": 900, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 8},
            {"iterations": 1200, "learning_rate": 0.03, "depth": 7, "l2_leaf_reg": 10},
        ]
        catboost_results = []
        for candidate_id, parameters in enumerate(catboost_candidates, start=1):
            candidate_model = CatBoostRegressor(
                **parameters, loss_function="MAE", eval_metric="MAE",
                has_time=True, one_hot_max_size=10, max_ctr_complexity=2,
                random_seed=SEED, thread_count=2, od_type="Iter", od_wait=80,
                allow_writing_files=False, verbose=False,
            )
            candidate_model.fit(
                train_pool, eval_set=validation_pool, use_best_model=True
            )
            prediction = candidate_model.predict(validation_pool)
            score = compact_score(segment_metrics(
                validation[TARGET], prediction, "catboost", "validation"
            ))
            catboost_results.append((score["combined_MAE_score"],
                                     score["global_MAE"], candidate_model,
                                     prediction, parameters, candidate_model.get_best_iteration()))
        best_catboost = min(catboost_results, key=lambda item: (item[0], item[1]))
        catboost_model = best_catboost[2]
        validation_predictions["catboost"] = best_catboost[3]
        SELECTED_HYPERPARAMETERS["catboost"] = {
            **best_catboost[4], "best_iteration": int(best_catboost[5])
        }
        catboost_hyperparameter_selection = pd.DataFrame([
            {"candidate": index + 1, "parameters": json.dumps(item[4]),
             "best_iteration": item[5], "combined_MAE_score": item[0],
             "global_MAE": item[1]}
            for index, item in enumerate(catboost_results)
        ]).sort_values(["combined_MAE_score", "global_MAE"])
        catboost_hyperparameter_selection.to_csv(
            REPORT_ROOT / "catboost_hyperparameter_selection.csv", index=False
        )
        display(catboost_hyperparameter_selection)
    except ImportError:
        print("CatBoost unavailable; install project requirements.")

## 6. Freeze winner on validation

Ranking gives equal weight to global MAE and MAE among flights delayed over 15
minutes. RMSE, median and p90 errors remain in the detailed segment report.
No test label is accessed here.

In [ ]:
if RUN_MODELS:
    validation_metrics = pd.concat([
        segment_metrics(validation[TARGET], prediction, name, "validation")
        for name, prediction in validation_predictions.items()
    ], ignore_index=True)
    validation_summary = pd.DataFrame([
        {"candidate": name, **compact_score(group)}
        for name, group in validation_metrics.groupby("model")
    ]).sort_values(["combined_MAE_score", "global_MAE"])
    validation_metrics.to_csv(
        REPORT_ROOT / "validation_segment_metrics.csv", index=False
    )
    validation_haul_direction_metrics = pd.concat([
        haul_direction_segment_metrics(
            validation, validation[TARGET], prediction, name, "validation"
        )
        for name, prediction in validation_predictions.items()
    ], ignore_index=True)
    validation_haul_direction_metrics.to_csv(
        REPORT_ROOT / "validation_haul_direction_metrics.csv", index=False
    )
    validation_summary.to_csv(
        REPORT_ROOT / "validation_model_ranking.csv", index=False
    )
    validation_summary[
        validation_summary["candidate"].isin(
            ["ridge", "ridge_without_registration"]
        )
    ].to_csv(REPORT_ROOT / "ridge_registration_ablation.csv", index=False)
    SELECTED_MODEL = validation_summary.iloc[0]["candidate"]
    (REPORT_ROOT / "selection.json").write_text(
        json.dumps({
            "selected_model": SELECTED_MODEL,
            "selected_hyperparameters": SELECTED_HYPERPARAMETERS.get(
                SELECTED_MODEL, {}
            ),
        }, indent=2),
        encoding="utf-8",
    )
    display(validation_summary)

## 7. Parallel delay classification

Regression remains the primary minute estimate. This additional task answers a
second operational question: will arrival delay exceed 15 minutes? The label is
derived only from the target and never enters the feature matrix. Accuracy is
reported, but model selection uses average precision because delayed flights are
the minority class. Balanced accuracy, precision, recall, F1, ROC-AUC, PR-AUC
and the complete confusion matrix prevent a majority-class prediction from looking
artificially strong. Logistic regularisation and two CatBoost configurations
are selected using December only. The 15-minute threshold is configurable above.


In [ ]:
if RUN_MODELS and RUN_CLASSIFICATION:
    train_delay_label = (train[TARGET] > DELAY_THRESHOLD_MINUTES).astype(int)
    validation_delay_label = (
        validation[TARGET] > DELAY_THRESHOLD_MINUTES
    ).astype(int)
    train_delay_rate = float(train_delay_label.mean())
    classification_probabilities = {
        "majority_baseline": np.full(len(validation), train_delay_rate)
    }

    for large_matrix in ("x_train", "x_validation", "x_tree_train",
                         "x_tree_validation"):
        globals().pop(large_matrix, None)
    gc.collect()
    classifier_preprocessor = MixedCategoricalRidgePreprocessor(
        LOW_CARDINALITY_COLUMNS, HIGH_CARDINALITY_COLUMNS, NUMERIC_COLUMNS
    )
    x_classifier_train = classifier_preprocessor.fit_transform(train)
    x_classifier_validation = classifier_preprocessor.transform(validation)
    logistic_candidates = {}
    for c_value in (0.05, 0.2, 1.0):
        classifier = LogisticRegression(
            C=c_value, solver="saga", max_iter=300, random_state=SEED,
        )
        classifier.fit(x_classifier_train, train_delay_label)
        probability = classifier.predict_proba(x_classifier_validation)[:, 1]
        name = f"logistic_c{c_value:g}"
        logistic_candidates[name] = classifier
        classification_probabilities[name] = probability
    del x_classifier_train, x_classifier_validation
    gc.collect()

    try:
        if not RUN_CATBOOST_CLASSIFIER:
            raise ImportError("CatBoost classifier disabled by configuration")
        from catboost import CatBoostClassifier, Pool
        classification_train_pool = Pool(
            cat_train[cat_features], train_delay_label.loc[cat_train.index],
            cat_features=CATEGORICAL_COLUMNS,
        )
        classification_validation_pool = Pool(
            cat_validation[cat_features],
            validation_delay_label.loc[cat_validation.index],
            cat_features=CATEGORICAL_COLUMNS,
        )
        catboost_classifier_candidates = [
            {"iterations": 700, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 8},
            {"iterations": 900, "learning_rate": 0.035, "depth": 7, "l2_leaf_reg": 10},
        ]
        catboost_classifier_results = []
        for candidate_id, parameters in enumerate(
            catboost_classifier_candidates, start=1
        ):
            candidate_classifier = CatBoostClassifier(
                **parameters, loss_function="Logloss", eval_metric="PRAUC",
                has_time=True, one_hot_max_size=10, max_ctr_complexity=2,
                random_seed=SEED, thread_count=2, od_type="Iter", od_wait=60,
                allow_writing_files=False, verbose=False,
            )
            candidate_classifier.fit(
                classification_train_pool,
                eval_set=classification_validation_pool, use_best_model=True,
            )
            probability = candidate_classifier.predict_proba(
                classification_validation_pool
            )[:, 1]
            metrics = delay_classification_metrics(
                validation[TARGET], probability, "catboost_classifier",
                "validation", delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            ).iloc[0]
            catboost_classifier_results.append((
                metrics["average_precision"], metrics["f1"],
                metrics["balanced_accuracy"], candidate_classifier,
                probability, parameters, candidate_classifier.get_best_iteration(),
            ))
        best_catboost_classifier = max(
            catboost_classifier_results, key=lambda item: (item[0], item[1], item[2])
        )
        catboost_classifier = best_catboost_classifier[3]
        classification_probabilities["catboost_classifier"] = best_catboost_classifier[4]
        SELECTED_HYPERPARAMETERS["catboost_classifier"] = {
            **best_catboost_classifier[5],
            "best_iteration": int(best_catboost_classifier[6]),
        }
        pd.DataFrame([
            {"candidate": index + 1, "parameters": json.dumps(item[5]),
             "best_iteration": item[6], "average_precision": item[0],
             "f1": item[1], "balanced_accuracy": item[2]}
            for index, item in enumerate(catboost_classifier_results)
        ]).sort_values("average_precision", ascending=False).to_csv(
            REPORT_ROOT / "catboost_classifier_hyperparameter_selection.csv",
            index=False,
        )
    except (ImportError, NameError):
        print("CatBoost classification unavailable; logistic models remain valid.")

    classification_validation_metrics = pd.concat([
        delay_classification_metrics(
            validation[TARGET], probability, name, "validation",
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        )
        for name, probability in classification_probabilities.items()
    ], ignore_index=True)
    regression_as_classification = delay_classification_metrics(
        validation[TARGET], validation_predictions[SELECTED_MODEL],
        f"{SELECTED_MODEL}_minutes_threshold", "validation",
        delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        scores_are_probabilities=False,
    )
    classification_validation_metrics = pd.concat(
        [classification_validation_metrics, regression_as_classification],
        ignore_index=True,
    )
    eligible_classifiers = classification_validation_metrics[
        classification_validation_metrics["model"] !=
        f"{SELECTED_MODEL}_minutes_threshold"
    ]
    SELECTED_CLASSIFIER = eligible_classifiers.sort_values(
        ["average_precision", "f1", "balanced_accuracy"],
        ascending=False,
    ).iloc[0]["model"]
    if SELECTED_CLASSIFIER in logistic_candidates:
        SELECTED_HYPERPARAMETERS[SELECTED_CLASSIFIER] = {
            "C": float(SELECTED_CLASSIFIER.removeprefix("logistic_c")),
            "solver": "saga",
        }
    classification_validation_metrics.to_csv(
        REPORT_ROOT / "classification_validation_metrics.csv", index=False
    )
    classification_validation_metrics[
        classification_validation_metrics["model"].str.startswith("logistic_")
    ].sort_values("average_precision", ascending=False).to_csv(
        REPORT_ROOT / "logistic_hyperparameter_selection.csv", index=False
    )
    validation_classification_by_haul_direction = pd.concat([
        haul_direction_classification_metrics(
            validation, validation[TARGET], probability, name, "validation",
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        )
        for name, probability in classification_probabilities.items()
    ], ignore_index=True)
    validation_classification_by_haul_direction.to_csv(
        REPORT_ROOT / "classification_validation_haul_direction_metrics.csv",
        index=False,
    )
    display(classification_validation_metrics.sort_values(
        "average_precision", ascending=False
    ))
    print({"selected_classifier": SELECTED_CLASSIFIER,
           "train_delay_rate": train_delay_rate})


## 8. Locked March and June evaluation

Run only after validation freezes the winner. March and June are reported
separately to expose temporal drift. A disappointing test must not be folded
back into training and retuned.

In [ ]:
def predict_frozen(name, frame):
    if name == "historical_baseline":
        return baseline.predict(frame)
    if name in ridge_models:
        return ridge_models[name].predict(ridge_preprocessors[name].transform(frame))
    if name in tree_models:
        matrix, _ = dense_tree_frame(frame, tree_medians)
        return tree_models[name].predict(matrix)
    if name == "catboost":
        prepared = frame.copy()
        prepared[CATEGORICAL_COLUMNS] = (
            prepared[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
        )
        prepared[NUMERIC_COLUMNS] = prepared[NUMERIC_COLUMNS].fillna(cat_medians)
        return catboost_model.predict(
            prepared[CATEGORICAL_COLUMNS + NUMERIC_COLUMNS]
        )
    raise KeyError(name)

if RUN_MODELS:
    locked_splits = {
        name: add_schedule_features(pd.read_parquet(DATA_ROOT / name))
        for name in ("test", "future_test")
    }
    assert all(not (forbidden & set(frame.columns)) for frame in locked_splits.values())
    final_metrics = pd.concat([
        segment_metrics(
            locked_splits[split_name][TARGET],
            predict_frozen(SELECTED_MODEL, locked_splits[split_name]),
            SELECTED_MODEL, split_name,
        )
        for split_name in ("test", "future_test")
    ], ignore_index=True)
    final_metrics.to_csv(REPORT_ROOT / "locked_test_metrics.csv", index=False)
    locked_haul_direction_metrics = pd.concat([
        haul_direction_segment_metrics(
            locked_splits[split_name], locked_splits[split_name][TARGET],
            predict_frozen(SELECTED_MODEL, locked_splits[split_name]),
            SELECTED_MODEL, split_name,
        )
        for split_name in ("test", "future_test")
    ], ignore_index=True)
    locked_haul_direction_metrics.to_csv(
        REPORT_ROOT / "locked_test_haul_direction_metrics.csv", index=False
    )
    display(final_metrics)

## 9. Save minute-regression contract

The bundle records the T-60 horizon, features, temporal periods and sample
fractions so later predictions cannot silently use a different contract.

In [ ]:
if RUN_MODELS:
    bundle = {
        "selected_model_name": SELECTED_MODEL,
        "prediction_horizon_minutes": 60,
        "target": TARGET,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "numeric_columns": NUMERIC_COLUMNS,
        "selected_hyperparameters": SELECTED_HYPERPARAMETERS.get(SELECTED_MODEL, {}),
        "train_sample_percent": TRAIN_SAMPLE_PERCENT,
        "tree_sample_percent": TREE_SAMPLE_PERCENT,
        "validation_period": "2022-12",
        "test_period": "2023-03",
        "future_test_period": "2023-06",
    }
    if SELECTED_MODEL in ridge_models:
        bundle.update({
            "model": ridge_models[SELECTED_MODEL],
            "preprocessor": ridge_preprocessors[SELECTED_MODEL],
            "categorical_columns": (
                LOW_CARDINALITY_COLUMNS + ridge_variant_columns[SELECTED_MODEL]
            ),
        })
    elif SELECTED_MODEL == "historical_baseline":
        bundle["model"] = baseline
    elif SELECTED_MODEL == "catboost":
        bundle.update({"model": catboost_model, "numeric_medians": cat_medians})
    else:
        bundle.update({"model": tree_models[SELECTED_MODEL],
                       "numeric_medians": tree_medians})
    path = MODEL_ROOT / "arrival_pre_t60_expanded_selected.joblib"
    joblib.dump(bundle, path)
    print({"saved": str(path)})

## 10. Locked classification tests and joint predictions

The classifier selected on December is frozen before March and June are opened.
Each output row contains both predictions: expected arrival-delay minutes and the
probability/binary decision for delay above the configured threshold.


In [ ]:
def predict_classifier_frozen(name, frame):
    if name == "majority_baseline":
        return np.full(len(frame), train_delay_rate)
    if name in logistic_candidates:
        matrix = classifier_preprocessor.transform(frame)
        return logistic_candidates[name].predict_proba(matrix)[:, 1]
    if name == "catboost_classifier":
        prepared = frame.copy()
        prepared[CATEGORICAL_COLUMNS] = (
            prepared[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
        )
        prepared[NUMERIC_COLUMNS] = prepared[NUMERIC_COLUMNS].fillna(cat_medians)
        return catboost_classifier.predict_proba(
            prepared[CATEGORICAL_COLUMNS + NUMERIC_COLUMNS]
        )[:, 1]
    raise KeyError(name)

if RUN_MODELS and RUN_CLASSIFICATION:
    locked_classification_rows = []
    locked_classification_segment_rows = []
    for split_name, frame in locked_splits.items():
        minute_prediction = predict_frozen(SELECTED_MODEL, frame)
        delay_probability = predict_classifier_frozen(SELECTED_CLASSIFIER, frame)
        locked_classification_rows.append(delay_classification_metrics(
            frame[TARGET], delay_probability, SELECTED_CLASSIFIER, split_name,
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        ))
        locked_classification_segment_rows.append(
            haul_direction_classification_metrics(
                frame, frame[TARGET], delay_probability,
                SELECTED_CLASSIFIER, split_name,
                delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            )
        )
        locked_classification_rows.append(delay_classification_metrics(
            frame[TARGET], minute_prediction,
            f"{SELECTED_MODEL}_minutes_threshold", split_name,
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            scores_are_probabilities=False,
        ))
        joint_predictions = pd.DataFrame({
            "ECTRL_ID": frame["ECTRL ID"].to_numpy(),
            "filed_off_block_time": frame["FILED OFF BLOCK TIME"].to_numpy(),
            "actual_arrival_delay_min": frame[TARGET].to_numpy(),
            "predicted_arrival_delay_min": minute_prediction,
            "actual_delayed_over_threshold": (
                frame[TARGET].to_numpy() > DELAY_THRESHOLD_MINUTES
            ),
            "predicted_delay_probability": delay_probability,
            "predicted_delayed_over_threshold": delay_probability >= 0.5,
            "duration_band": frame["Duration_Band"].to_numpy(),
            "transatlantic_direction": frame["Transatlantic_Direction"].to_numpy(),
        })
        joint_predictions.to_parquet(
            REPORT_ROOT / f"{split_name}_joint_predictions.parquet", index=False
        )
    locked_classification_metrics = pd.concat(
        locked_classification_rows, ignore_index=True
    )
    locked_classification_metrics.to_csv(
        REPORT_ROOT / "classification_locked_test_metrics.csv", index=False
    )
    pd.concat(locked_classification_segment_rows, ignore_index=True).to_csv(
        REPORT_ROOT / "classification_locked_test_haul_direction_metrics.csv",
        index=False,
    )
    display(locked_classification_metrics)


## 11. Save classification contract

The classification artefact is separate from the minute-regression artefact, so
either task can be updated or deployed without deleting the other.


In [ ]:
if RUN_MODELS and RUN_CLASSIFICATION:
    classifier_bundle = {
        "selected_model_name": SELECTED_CLASSIFIER,
        "prediction_horizon_minutes": 60,
        "target_definition": f"{TARGET} > {DELAY_THRESHOLD_MINUTES:g}",
        "delay_threshold_minutes": DELAY_THRESHOLD_MINUTES,
        "probability_threshold": 0.5,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "numeric_columns": NUMERIC_COLUMNS,
        "selected_hyperparameters": SELECTED_HYPERPARAMETERS.get(SELECTED_CLASSIFIER, {}),
        "train_sample_percent": TRAIN_SAMPLE_PERCENT,
        "validation_period": "2022-12",
        "test_period": "2023-03",
        "future_test_period": "2023-06",
    }
    if SELECTED_CLASSIFIER == "majority_baseline":
        classifier_bundle["train_delay_rate"] = train_delay_rate
    elif SELECTED_CLASSIFIER in logistic_candidates:
        classifier_bundle.update({
            "model": logistic_candidates[SELECTED_CLASSIFIER],
            "preprocessor": classifier_preprocessor,
        })
    else:
        classifier_bundle.update({
            "model": catboost_classifier,
            "numeric_medians": cat_medians,
        })
    classifier_path = MODEL_ROOT / "arrival_pre_t60_expanded_classifier.joblib"
    joblib.dump(classifier_bundle, classifier_path)
    print({"saved_classifier": str(classifier_path)})


## Guardrails

- Lower validation error does not prove future reliability; report both tests.
- The new months improve coverage but remain non-consecutive snapshots.
- Weather stays deferred until this expanded flight-only baseline is frozen.
- Classification means arrival delay >15 minutes; exactly 15 remains OTP15.
- Accuracy must be read with recall, precision and PR-AUC because classes are imbalanced.
- CNN/LSTM remains secondary until dense continuous sequences are available.